In [1]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Only GPU 0 is visible

In [2]:
import torch
import torch.nn as nn
from transformers import CLIPVisionModel, AutoModel


In [3]:
#BAAI/EVA-CLIP-L-14, openai/clip-vit-base-patch32

In [4]:
class VisionTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")

    def forward(self, pixel_values):
        return self.vision(pixel_values).pooler_output


In [5]:
class TextTower(nn.Module):
    def __init__(self):
        super().__init__()
        # Use AutoModel which is already imported and works for DistilBERT
        self.text = AutoModel.from_pretrained("distilbert-base-uncased")

    def forward(self, input_ids, attention_mask):
        # Return CLS token output
        return self.text(input_ids, attention_mask).last_hidden_state[:,0]

In [6]:
class VLM(nn.Module):
    def __init__(self, num_answers):
        super().__init__()

        self.vision = VisionTower()
        self.text   = TextTower()

        self.fusion = nn.Sequential(
            nn.Linear(768 * 2, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),#change second value here for dimension
            nn.ReLU()
        )

        self.classifier = nn.Linear(512, num_answers) #change first value

    def forward(self, pixel_values, input_ids, attention_mask):

        v = self.vision(pixel_values)
        t = self.text(input_ids, attention_mask)

        fused = torch.cat([v, t], dim=1)

        z = self.fusion(fused)

        logits = self.classifier(z)

        return logits, z


In [7]:
from transformers import CLIPProcessor, AutoTokenizer
from transformers import SiglipVisionModel,AutoProcessor
image_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
#image_processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (2 outputs left out)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
from datasets import load_dataset
from torch.utils.data import ConcatDataset
#chandrabhuma/csi-vqa-dataset_i2_28b
#chandrabhuma/csi-vqa-city2chicago28
#chandrabhuma/csi-vqa-city1losangeles28
#chandrabhuma/csi-vqa-city11santaclara28
#chandrabhuma/csi-vqa-city8dallas3p5
#chandrabhuma/csi-vqa-o1drone200
#chandrabhuma/csi-vqa-santaclara28Hmag
#chandrabhuma/csi-vqa-santaclara28Subcar128
train_dataset = load_dataset("chandrabhuma/csi-vqa-santaclara28SNRZERO", split = "train")
test_dataset = load_dataset("chandrabhuma/csi-vqa-santaclara28SNRZERO", split = "test")


README.md:   0%|          | 0.00/506 [00:00<?, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/471M [00:00<?, ?B/s]

data/train-00002-of-00006.parquet:   0%|          | 0.00/465M [00:00<?, ?B/s]

data/train-00003-of-00006.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

data/train-00004-of-00006.parquet:   0%|          | 0.00/467M [00:00<?, ?B/s]

data/train-00005-of-00006.parquet:   0%|          | 0.00/466M [00:00<?, ?B/s]

data/test-00000-of-00002.parquet:   0%|          | 0.00/350M [00:00<?, ?B/s]

data/test-00001-of-00002.parquet:   0%|          | 0.00/353M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [9]:
#train_dataset=train_dataset1.filter(lambda x: x["question_type"] == "multipath_structure")
#test_dataset=test_dataset1.filter(lambda x: x["question_type"] == "multipath_structure")

In [10]:
print(len(train_dataset))
print(len(test_dataset))

40000
10000


In [11]:
from torch.utils.data import Dataset

class ChessVQADataset(Dataset):
    def __init__(self, hf_dataset, answer2id):
        self.dataset = hf_dataset
        self.answer2id = answer2id

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        image = sample["image"]
        question = sample["question"]
        answer = sample["answer"]

        label = self.answer2id[answer]

        return image, question, label


In [12]:
from transformers import CLIPProcessor, AutoTokenizer



def collate_fn(batch):
    images, questions, answers = zip(*batch)

    pixel_values = image_processor(
        list(images),
        return_tensors="pt"
    ).pixel_values

    text_inputs = tokenizer(
        list(questions),
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    labels = torch.tensor(answers, dtype=torch.long)

    return pixel_values, text_inputs, labels



In [13]:
# # Build answer vocabulary from both train and test ....single dataset
all_answers = sorted(list(set(train_dataset['answer']) | set(test_dataset['answer'])))
answer2id = {a: i for i, a in enumerate(all_answers)}
id2answer = {i: a for a, i in answer2id.items()}

num_answers = len(answer2id)
print("Number of unique answers (combined):", num_answers)

Number of unique answers (combined): 39


In [14]:
all_answers

['0',
 '10',
 '11',
 '16',
 '17',
 '18',
 '19',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '3',
 '30',
 '31',
 '32',
 '4',
 '5',
 '6',
 '7',
 '9',
 'FewPaths',
 'High',
 'Low',
 'Mild',
 'Moderate',
 'No',
 'Region1',
 'Region2',
 'Region3',
 'Region4',
 'RichMultipath',
 'Severe',
 'Yes']

In [15]:
# Optimized: build answer vocabulary by accessing underlying HF datasets metadata
# all_answers_set = set()

# # train_dataset and test_dataset are ConcatDatasets, we iterate through their internal datasets
# for ds in train_dataset.datasets:
#     all_answers_set.update([str(a) for a in ds['answer']])

# for ds in test_dataset.datasets:
#     all_answers_set.update([str(a) for a in ds['answer']])

# all_answers = sorted(list(all_answers_set))
# answer2id = {a: i for i, a in enumerate(all_answers)}
# id2answer = {i: a for a, i in answer2id.items()}

# num_answers = len(answer2id)
# print("Number of unique answers (combined):", num_answers)

In [16]:
from torch.utils.data import DataLoader

# Re-instantiate datasets and loaders with the updated answer2id
train_torch_dataset = ChessVQADataset(train_dataset, answer2id)
test_torch_dataset  = ChessVQADataset(test_dataset,  answer2id)

train_loader = DataLoader(
    train_torch_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_torch_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"



In [18]:
# 1. Instantiate model with correct num_answers
model = VLM(num_answers).to(device)

# 2. Freeze pretrained towers
for p in model.vision.parameters():
    p.requires_grad = False

for p in model.text.parameters():
    p.requires_grad = False

# 3. Define optimizer
optimizer = torch.optim.AdamW(
    list(model.fusion.parameters()) +
    list(model.classifier.parameters()),
    lr=1e-3
)

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias          

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
#beam_prediction(80%), mimo_property(94.52%),channel_property(100%)
#antenna_gain..bad, subcarrier_fading...bad

In [20]:
# 3. Training Loop
model.train()
for epoch in range(5):
  for step, (pixel_values, text_inputs, labels) in enumerate(train_loader):
      pixel_values = pixel_values.to(device)
      text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
      labels = labels.to(device)

      logits, z = model(
          pixel_values,
          text_inputs["input_ids"],
          text_inputs["attention_mask"]
      )

      loss = nn.CrossEntropyLoss()(logits, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      if step % 50 == 0:
          print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")

Epoch 0 | Step 0 | Loss: 3.6623
Epoch 0 | Step 50 | Loss: 1.9936
Epoch 0 | Step 100 | Loss: 0.9743
Epoch 0 | Step 150 | Loss: 0.2265
Epoch 0 | Step 200 | Loss: 0.5139
Epoch 0 | Step 250 | Loss: 0.3535
Epoch 0 | Step 300 | Loss: 0.2790
Epoch 0 | Step 350 | Loss: 0.0580
Epoch 0 | Step 400 | Loss: 0.4608
Epoch 0 | Step 450 | Loss: 0.0658
Epoch 0 | Step 500 | Loss: 0.5237
Epoch 0 | Step 550 | Loss: 0.1403
Epoch 0 | Step 600 | Loss: 0.3672
Epoch 0 | Step 650 | Loss: 0.2810
Epoch 0 | Step 700 | Loss: 0.0760
Epoch 0 | Step 750 | Loss: 0.2102
Epoch 0 | Step 800 | Loss: 0.1892
Epoch 0 | Step 850 | Loss: 0.2340
Epoch 0 | Step 900 | Loss: 0.1009
Epoch 0 | Step 950 | Loss: 0.2397
Epoch 0 | Step 1000 | Loss: 0.4383
Epoch 0 | Step 1050 | Loss: 0.2331
Epoch 0 | Step 1100 | Loss: 0.0575
Epoch 0 | Step 1150 | Loss: 0.0285
Epoch 0 | Step 1200 | Loss: 0.1808
Epoch 1 | Step 0 | Loss: 0.2488
Epoch 1 | Step 50 | Loss: 0.0582
Epoch 1 | Step 100 | Loss: 0.2405
Epoch 1 | Step 150 | Loss: 0.2563
Epoch 1 | Step 

In [21]:
print("Vision trainable:", sum(p.numel() for p in model.vision.parameters() if p.requires_grad))
print("Text trainable:",   sum(p.numel() for p in model.text.parameters() if p.requires_grad))
print("Fusion trainable:", sum(p.numel() for p in model.fusion.parameters() if p.requires_grad))
print("Classifier trainable:", sum(p.numel() for p in model.classifier.parameters() if p.requires_grad))



Vision trainable: 0
Text trainable: 0
Fusion trainable: 2098688
Classifier trainable: 20007


In [22]:
import torch
from tqdm import tqdm

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for pixel_values, text_inputs, labels in tqdm(test_loader):

        pixel_values = pixel_values.to(device)
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
        labels = labels.to(device)

        logits, z = model(
            pixel_values,
            text_inputs["input_ids"],
            text_inputs["attention_mask"]
        )

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total * 100

print(f"\nTest Accuracy: {accuracy:.2f}%")

100%|██████████| 313/313 [00:17<00:00, 18.16it/s]


Test Accuracy: 96.44%


In [23]:
model.eval()

with torch.no_grad():
    for batch_idx, (pixel_values, text_inputs, labels) in enumerate(test_loader):

        pixel_values = pixel_values.to(device)
        text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
        labels = labels.to(device)

        logits, _ = model(
            pixel_values,
            text_inputs["input_ids"],
            text_inputs["attention_mask"]
        )

        preds = torch.argmax(logits, dim=1)

        for i in range(min(5, labels.size(0))):
            gt_id = labels[i].item()
            pred_id = preds[i].item()

            print(f"Sample {i}")
            print("Question:", test_dataset[batch_idx * test_loader.batch_size + i]["question"])
            print("GT Answer:", id2answer[gt_id])
            print("Predicted:", id2answer[pred_id])
            print("-" * 60)

        break   # only first batch


Sample 0
Question: What is the channel rank?
GT Answer: 32
Predicted: 32
------------------------------------------------------------
Sample 1
Question: Is antenna correlation high or low?
GT Answer: Low
Predicted: Low
------------------------------------------------------------
Sample 2
Question: What is the spatial correlation level?
GT Answer: High
Predicted: High
------------------------------------------------------------
Sample 3
Question: What is the channel rank?
GT Answer: 32
Predicted: 32
------------------------------------------------------------
Sample 4
Question: Which beam region contains the optimal beam?
GT Answer: Region4
Predicted: Region4
------------------------------------------------------------
